# Signal Quality Analysis
Load saved waveform artifacts and compute Pearson, DTW, SNR, and PSD correlation.

In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import torch

try:
    from scipy.signal import welch
except Exception:
    welch = None

In [ ]:
# Update this path to your experiment output
WAVEFORM_PATH = Path('runs/exp/SSMamba_xxx/signal_quality/waveforms.pt')
PRED_KEY = 'pred_norm'  # or 'pred_raw'
DTW_STEP = 2

In [ ]:
def to_1d_numpy(x):
    return x.detach().cpu().numpy().astype(np.float64).reshape(-1)

def reconstruct_subject_signal(subject_chunks):
    keys = sorted(subject_chunks.keys())
    chunks = [to_1d_numpy(subject_chunks[k]) for k in keys]
    return np.concatenate(chunks) if chunks else np.array([], dtype=np.float64)

def pearson_corr(x, y):
    n = min(len(x), len(y))
    if n == 0:
        return np.nan
    x = x[:n]
    y = y[:n]
    if np.std(x) < 1e-12 or np.std(y) < 1e-12:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])

def dtw_distance(x, y):
    n, m = len(x), len(y)
    if n == 0 or m == 0:
        return np.nan
    dp = np.full((n + 1, m + 1), np.inf, dtype=np.float64)
    dp[0, 0] = 0.0
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = abs(x[i - 1] - y[j - 1])
            dp[i, j] = cost + min(dp[i - 1, j], dp[i, j - 1], dp[i - 1, j - 1])
    return float(dp[n, m])

def projection_snr_db(pred, gt):
    n = min(len(pred), len(gt))
    if n == 0:
        return np.nan
    p, g = pred[:n], gt[:n]
    den = float(np.dot(g, g))
    if den < 1e-12:
        return np.nan
    a = float(np.dot(p, g) / den)
    signal = a * g
    noise = p - signal
    p_sig = float(np.mean(signal ** 2))
    p_noise = float(np.mean(noise ** 2))
    if p_noise < 1e-12:
        return np.inf
    if p_sig < 1e-12:
        return -np.inf
    return float(10.0 * math.log10(p_sig / p_noise))

def compute_psd(x, fs):
    n = len(x)
    if n == 0:
        return np.array([]), np.array([])
    if welch is not None:
        return welch(x, fs=fs, nperseg=min(256, n))
    x = x - np.mean(x)
    fft = np.fft.rfft(x)
    pxx = (np.abs(fft) ** 2) / max(n, 1)
    f = np.fft.rfftfreq(n, d=1.0 / fs)
    return f, pxx

def psd_corr(pred, gt, fs, fmin=0.7, fmax=4.0):
    fp, pp = compute_psd(pred, fs)
    fg, pg = compute_psd(gt, fs)
    if len(fp) == 0 or len(fg) == 0:
        return np.nan
    f = fp if len(fp) <= len(fg) else fg
    pp_i = np.interp(f, fp, pp)
    pg_i = np.interp(f, fg, pg)
    band = (f >= fmin) & (f <= fmax)
    if not np.any(band):
        return np.nan
    a = np.log(pp_i[band] + 1e-12)
    b = np.log(pg_i[band] + 1e-12)
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])

In [ ]:
blob = torch.load(WAVEFORM_PATH, map_location='cpu')
pred_dict = blob[PRED_KEY]
gt_dict = blob['gt']
fs = float(blob.get('fs', 30.0))

rows = []
for subj in sorted(set(pred_dict.keys()) & set(gt_dict.keys())):
    pred = reconstruct_subject_signal(pred_dict[subj])
    gt = reconstruct_subject_signal(gt_dict[subj])
    n = min(len(pred), len(gt))
    if n == 0:
        continue
    pred = pred[:n]
    gt = gt[:n]

    pred_d = pred[::max(DTW_STEP, 1)]
    gt_d = gt[::max(DTW_STEP, 1)]
    dtw = dtw_distance(pred_d, gt_d)

    rows.append({
        'subject': str(subj),
        'num_samples': int(n),
        'pearson': pearson_corr(pred, gt),
        'dtw': dtw,
        'dtw_norm': dtw / max(len(pred_d), len(gt_d), 1) if np.isfinite(dtw) else np.nan,
        'snr_db': projection_snr_db(pred, gt),
        'psd_corr': psd_corr(pred, gt, fs),
    })

df = pd.DataFrame(rows)
df

In [ ]:
summary = df[['pearson', 'dtw', 'dtw_norm', 'snr_db', 'psd_corr']].describe().T
summary

In [ ]:
out_dir = Path('signal')
out_dir.mkdir(exist_ok=True)
df.to_csv(out_dir / 'signal_quality_report.csv', index=False)
summary.to_csv(out_dir / 'signal_quality_summary.csv')
print('Saved:', out_dir / 'signal_quality_report.csv')
print('Saved:', out_dir / 'signal_quality_summary.csv')